<a href="https://colab.research.google.com/github/hd07042014-maker/Movement-tracker/blob/main/Cash_Pop.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [20]:
from collections import Counter
import re
import pandas as pd
from pypdf import PdfReader
from google.colab import files
import os

def select_pdf_file():
  """Prompts the user to upload a PDF file using Colab's native file upload tool."""
  print("Please upload your Cash Pop PDF File:")
  uploaded = files.upload()
  for filename in uploaded.keys():
    if filename.endswith(".pdf"):
      return filename
  return None

def extract_cash_pop_data(pdf_path):
  reader = PdfReader(pdf_path)
  full_text = ""
  for page in reader.pages:
    full_text += page.extract_text() + "\n"

  draws = []
  # Robust pattern to match the PDF text layout: DRAW # - <id> <date> <winning_number>
  pattern = re.compile(
      r"DRAW\s*#?\s*-?\s*(\d+)\s+(\d{2}/\d{2}/\d{4})\s+(\d+)", re.IGNORECASE
  )
  matches = pattern.findall(full_text)

  for match in matches:
    draw_id = int(match[0])
    date = match[1]
    winning_num = int(match[2])
    draws.append({"draw": draw_id, "date": date, "number": winning_num})

  return draws

def get_frequency_table(data, n):
  subset = data[:n]
  numbers = [item["number"] for item in subset]
  counts = Counter(numbers)

  # Ensure all numbers from 1 to 15 are included, defaulting to 0 for zero occurrences
  full_counts = {i: counts.get(i, 0) for i in range(1, 16)}

  df = pd.DataFrame(
      list(full_counts.items()), columns=["Winning Number", "Frequency"]
  )
  # Sort by Frequency descending, then by Winning Number ascending for ties
  df = df.sort_values(
      by=["Frequency", "Winning Number"], ascending=[False, True]
  ).reset_index(drop=True)
  return df

def main():
  pdf_path = select_pdf_file()

  if not pdf_path:
    print("No PDF file uploaded. Program terminated.")
    return

  print(f"\nProcessing file: {pdf_path}\n")
  draws_data = extract_cash_pop_data(pdf_path)

  # Clean up the uploaded file to keep the directory clean
  if os.path.exists(pdf_path):
    os.remove(pdf_path)

  if not draws_data:
    print("No valid draw data found in the selected PDF.")
    return

  # Sort descending by draw number to ensure newest draws come first
  draws_data = sorted(draws_data, key=lambda x: x["draw"], reverse=True)

  df_50 = get_frequency_table(draws_data, 50)
  df_100 = get_frequency_table(draws_data, 100)
  df_150 = get_frequency_table(draws_data, 150)

  print("--- Latest 50 Draws Frequency Table ---")
  print(df_50.to_string(index=False))

  print("\n--- Latest 100 Draws Frequency Table ---")
  print(df_100.to_string(index=False))

  print("\n--- Latest 150 Draws Frequency Table ---")
  print(df_150.to_string(index=False))

if __name__ == "__main__":
  main()

Please upload your Cash Pop PDF File:


Saving Cash Pop file.pdf to Cash Pop file.pdf

Processing file: Cash Pop file.pdf

--- Latest 50 Draws Frequency Table ---
 Winning Number  Frequency
             12          9
             15          5
              6          4
              9          4
             13          4
              1          3
              2          3
              3          3
              4          3
              5          3
              7          3
              8          3
             10          2
             11          1
             14          0

--- Latest 100 Draws Frequency Table ---
 Winning Number  Frequency
             12         13
             13         10
              1          9
              6          9
             15          9
              8          8
              7          7
              9          6
              2          5
              3          5
              4          5
              5          5
             14          5
             10          